In [13]:
# %pip menginstal ke environment KERNEL yang aktif — kompatibel di Colab, Jupyter, & VS Code.
%pip install trafilatura feedparser googlenewsdecoder aiohttp supabase python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [14]:
import re
import time
import random
import asyncio
import urllib.parse

import aiohttp
import feedparser
import trafilatura
from supabase import create_client
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from googlenewsdecoder import gnewsdecoder

In [15]:
PORTAL_TIER_1 = [
    "kompas.com",
    "detik.com",
    "tempo.co",
    "cnnindonesia.com",
    "cnbcindonesia.com",
    "antaranews.com",
    "liputan6.com",
    "republika.co.id",
    "suara.com",
    "tirto.id"
]

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.3 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_4_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4.1 Mobile/15E148 Safari/604.1"
]

In [ ]:
# Panjang minimal teks artikel (char) agar dianggap valid -> buang fragmen nav/menu.
MIN_PANJANG_ISI = 120


async def proses_artikel_async(session, item, sem, stat):
    """Concurrent per-artikel: decode URL Google News -> cek URL sampah -> fetch -> ekstrak teks.

    Mengembalikan `item` terisi (dict) bila sukses, atau None bila gagal.
    Sebab kegagalan dicatat ke dict `stat` agar yield 0 tidak misterius.
    """
    async with sem:
        await asyncio.sleep(random.uniform(0.5, 1.5))
        # 1) Decode redirect Google News -> URL portal asli (lib sinkron -> jalankan di thread).
        #    trafilatura TIDAK bisa mengekstrak dari link Google News, jadi decode wajib sukses.
        try:
            decode = await asyncio.to_thread(gnewsdecoder, item["google_link"])
            url_asli = decode.get("decoded_url") if decode.get("status") else None
        except Exception:
            url_asli = None
        if not url_asli:
            stat["gagal_decode"] += 1
            return None
        item["URL"] = url_asli

        # 2) Buang URL sampah (index/tag) yang lolos filter judul.
        if any(p in url_asli.lower() for p in pola_url_sampah):
            stat["sampah_url"] += 1
            return None

        # 3) Jeda non-blocking 0.5-1.5 detik (pola menyerupai manusia) lalu fetch + ekstrak.
        headers = {
            "User-Agent": random.choice(USER_AGENTS),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
            "Referer": "https://www.google.com/",
        }
        try:
            async with session.get(url_asli, headers=headers,
                                   timeout=aiohttp.ClientTimeout(total=15)) as response:
                if response.status != 200:
                    stat["gagal_http"] += 1
                    return None
                html = await response.text()
                # trafilatura.extract bersifat blocking -> jalankan di thread terpisah.
                teks = await asyncio.to_thread(trafilatura.extract, html)
        except Exception:
            stat["gagal_fetch"] += 1
            return None

        if not teks or len(teks) < MIN_PANJANG_ISI:
            stat["terlalu_pendek"] += 1
            return None

        item["Isi_Berita"] = teks
        return item

In [17]:
def hapus_watermark_portal(judul_bawaan_google, nama_portal):
    """Menghapus watermark ' - Nama Portal' yang ditambahkan Google News pada judul."""
    suffix_google = f" - {nama_portal}"
    if judul_bawaan_google.endswith(suffix_google):
        return judul_bawaan_google[:-len(suffix_google)].strip()
    # Fallback: buang potongan ' - ...' terakhir bila nama portal tidak sama persis.
    return re.sub(r'\s*-\s*[^-]+$', '', judul_bawaan_google).strip()

In [18]:
pola_mutlak = [
    # Detik & Kompas
    "berita dan informasi",
    "kabar akurat terpercaya",
    "timeline berita terbaru",
    "berita harian",

    # Liputan6, Suara, Tirto
    "kumpulan berita",
    "kumpulan artikel",
    "top 3 berita",

    # General Index Pages (Semua Portal)
    "indeks berita",
    "berita terpopuler",
    "top news",

    # Non-Berita / Tabular Data (Bikin AI bingung)
    "jadwal sholat",
    "prakiraan cuaca",
    "kurs valas"
]

pola_awalan = [
    # Awalan SEO Umum (Detik, Kompas, Republika)
    "berita terkini",
    "berita terbaru",
    "terkini dan terbaru",

    # Awalan Khas Liputan6, Tempo, Suara
    "berita hari ini",
    "kabar terbaru",
    "headline hari ini",

    # Awalan Khas CNN & CNBC
    "fokus berita",
    "kabar harian"
]

kata_haram = ["zodiak", "ramalan", "promo", "diskon", "lirik lagu"]

pola_url_sampah = ["/tag/", "/tags/", "/indeks/", "/index/"]

In [19]:
def apakah_berita_sampah(judul, url):
    """True bila judul/url terindikasi bukan berita substantif (index, tag, SEO filler)."""
    if not judul or not judul.strip():
        return True
    if len(judul.strip().split()) < 3:
        return True
    if any(p in url for p in pola_url_sampah):
        return True
    if any(kata in judul for kata in kata_haram):
        return True
    if any(pola in judul for pola in pola_mutlak):
        return True
    # Pengecekan presisi tinggi: hanya di awal judul.
    if any(judul.startswith(pola) for pola in pola_awalan):
        return True
    return False

In [20]:
# ==========================================
# WORKER 1: FETCHING BERITA
# ==========================================
async def _ambil_feed_rss(portal):
    """Ambil 1 feed RSS Google News untuk satu portal (feedparser blocking -> thread)."""
    query_encoded = urllib.parse.quote_plus(f"site:{portal} when:1h")
    url_rss = f"https://news.google.com/rss/search?q={query_encoded}&hl=id&gl=ID&ceid=ID:id"
    return await asyncio.to_thread(feedparser.parse, url_rss)


async def agregator_seluruh_berita_tier1_async():
    """Kumpulkan URL dari RSS Google News lalu decode + ekstrak teks secara paralel."""
    print(f"\n{'='*40}")
    print(f"🔄 BATCH: {datetime.now(ZoneInfo('Asia/Jakarta')).strftime('%H:%M:%S WIB')} | ASYNC SCRAPER")
    print(f"{'='*40}")

    stat = {"sampah_judul": 0, "gagal_decode": 0, "sampah_url": 0,
            "gagal_http": 0, "gagal_fetch": 0, "terlalu_pendek": 0}

    # FASE 1: Ambil SEMUA feed RSS paralel, lalu saring sampah berbasis JUDUL (tanpa jaringan).
    #         Decode URL yang mahal SENGAJA ditunda ke Fase 2 agar tidak terbuang untuk sampah.
    feeds = await asyncio.gather(*[_ambil_feed_rss(p) for p in PORTAL_TIER_1])

    items, link_terlihat = [], set()
    for feed in feeds:
        for entry in feed.entries:
            try:
                link = entry.link
                if link in link_terlihat:          # buang duplikat dalam-batch sedini mungkin
                    continue
                link_terlihat.add(link)

                nama_portal = entry.source.title if "source" in entry else "Tidak diketahui"
                judul = hapus_watermark_portal(entry.title, nama_portal)

                # Filter judul dulu (murah) -> hemat panggilan decode yang mahal.
                if apakah_berita_sampah(judul.lower(), ""):
                    stat["sampah_judul"] += 1
                    continue

                # published_parsed (UTC) bisa None -> fallback ke waktu sekarang (WIB).
                if entry.get("published_parsed"):
                    waktu = (datetime(*entry.published_parsed[:6]) + timedelta(hours=7)) \
                        .strftime("%Y-%m-%d %H:%M:%S")
                else:
                    waktu = datetime.now(ZoneInfo("Asia/Jakarta")).strftime("%Y-%m-%d %H:%M:%S")

                items.append({"Judul": judul, "Portal": nama_portal, "Waktu_Rilis": waktu,
                              "google_link": link, "URL": "", "Isi_Berita": ""})
            except Exception:
                continue

    if not items:
        print("INFO: Tidak ada berita baru.")
        return []

    # FASE 2: Decode + fetch + ekstraksi PARALEL (decode Google News kini ikut diparalelkan).
    sem = asyncio.Semaphore(10)   # batasi konkurensi agar Google/portal tidak memblokir burst
    async with aiohttp.ClientSession() as session:
        hasil = await asyncio.gather(
            *[proses_artikel_async(session, it, sem, stat) for it in items]
        )

    # FASE 3: Kumpulkan yang valid + dedup berdasarkan URL asli (otoritatif, cegah duplikat upsert).
    semua_berita_valid, url_terlihat = [], set()
    for r in hasil:
        if r and r["URL"] not in url_terlihat:
            url_terlihat.add(r["URL"])
            semua_berita_valid.append(r)

    semua_berita_valid.sort(key=lambda x: x["Waktu_Rilis"], reverse=True)

    print("--- HASIL PENARIKAN ASINKRON ---")
    print(f"Kandidat (judul lolos): {len(items)}  ->  Valid: {len(semua_berita_valid)}")
    print(f"  Dibuang: judul-sampah={stat['sampah_judul']} url-sampah={stat['sampah_url']} "
          f"decode-gagal={stat['gagal_decode']} http!=200={stat['gagal_http']} "
          f"fetch-gagal={stat['gagal_fetch']} terlalu-pendek={stat['terlalu_pendek']}")
    print(f"✅ SUKSES: {len(semua_berita_valid)} berita valid.")

    return semua_berita_valid

In [21]:
import os
from dotenv import load_dotenv

# Baca kredensial dari file .env (bukan hardcode). Scraper cukup pakai kunci anon.
load_dotenv()
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_ANON_KEY"]

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [22]:
async def simpan_berita_ke_db_async(array_berita_bersih):
    """
    Menyimpan berita ke Supabase secara batch & asinkron.
    Secara otomatis mengabaikan URL duplikat.
    """
    if not array_berita_bersih:
        return

    print(f"\n[INGESTION] Menyuntikkan {len(array_berita_bersih)} berita ke Supabase secara masal...")

    data_untuk_dimasukkan = [
        {
            "judul": b["Judul"],
            "isi_teks": b["Isi_Berita"],
            "portal_sumber": b["Portal"],
            "url_asli": b["URL"],
            "waktu_rilis": b["Waktu_Rilis"],
            "status_proses": 0
        }
        for b in array_berita_bersih
    ]

    # Wrapper fungsi sinkron Supabase agar tidak memblokir event loop
    def jalankan_upsert():
        return supabase.table("tabel_berita").upsert(
            data_untuk_dimasukkan,
            on_conflict="url_asli",
            ignore_duplicates=True # Abaikan baris baru jika URL sudah ada di DB
        ).execute()

    try:
        # Eksekusi upsert di thread terpisah
        hasil = await asyncio.to_thread(jalankan_upsert)

        berita_masuk = len(hasil.data)
        print(f"[INGESTION] Sukses! {berita_masuk} berita (baru) masuk ke Supabase.")
    except Exception as e:
        print(f"⚠️ Gagal melakukan Batch Upsert: {e}")

In [ ]:
async def run_worker_1_periodically():
    interval_detik = 900  # Target eksekusi pasti: 30 menit

    while True:
        waktu_mulai = time.time()

        # 1. Eksekusi scraper.
        data_berita = await agregator_seluruh_berita_tier1_async()

        for i, b in enumerate(data_berita[:5], start=1):
            print(f"{i:02d}. {b['Judul']} [{b['Waktu_Rilis']}]")

        # 2. Simpan ke Supabase secara asinkron.
        if data_berita:
            await simpan_berita_ke_db_async(data_berita)

        # 3. Hitung durasi dan sisa waktu tidur.
        durasi_eksekusi = time.time() - waktu_mulai
        waktu_tidur = interval_detik - durasi_eksekusi

        if waktu_tidur > 0:
            print(f"\n[✅ Siklus Selesai ({durasi_eksekusi:.2f} detik). Tidur selama {waktu_tidur:.2f} detik...]")
            await asyncio.sleep(waktu_tidur)
        else:
            print(f"\n[⚠️ Bahaya: Eksekusi melampaui interval 30 menit ({durasi_eksekusi:.2f} detik). Langsung gas siklus baru!]")

In [24]:
if __name__ == "__main__":
    # Karena Jupyter sudah memiliki event loop yang berjalan, panggil fungsi asinkron dengan await
    await run_worker_1_periodically()


🔄 BATCH: 23:44:31 WIB | ASYNC SCRAPER
--- HASIL PENARIKAN ASINKRON ---
Kandidat (judul lolos): 176  ->  Valid: 166
  Dibuang: judul-sampah=62 url-sampah=3 decode-gagal=0 http!=200=5 fetch-gagal=0 terlalu-pendek=2
✅ SUKSES: 166 berita valid.
01. Penyelidikan Tak Berhenti, KPK Cari Celah Dugaan Korupsi MBG [2026-06-22 23:40:46]
02. Jadwal Siaran Langsung Perancis Vs Irak Piala Dunia 2026, Selasa 23 Juni Pukul 04.00 WIB [2026-06-22 23:38:35]
03. Ibu-ibu PKK Ikut Demonstrasi: Kami Dukung MBG untuk Balita [2026-06-22 23:36:11]
04. Ternyata Segini Kebun Binatang Ragunan [2026-06-22 23:36:00]
05. Jadwal Piala Dunia 23-24 Juni 2026 [2026-06-22 23:35:21]

[INGESTION] Menyuntikkan 166 berita ke Supabase secara masal...
[INGESTION] Sukses! 130 berita (baru) masuk ke Supabase.

[✅ Siklus Selesai (42.53 detik). Tidur selama 857.47 detik...]

🔄 BATCH: 23:59:31 WIB | ASYNC SCRAPER
--- HASIL PENARIKAN ASINKRON ---
Kandidat (judul lolos): 152  ->  Valid: 139
  Dibuang: judul-sampah=69 url-sampah=3 dec

CancelledError: 